# 03 · Are the synergies stable and real?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 1 · Existence: is the EMG low-dimensional beyond chance?**
- **Stage 2 · Artefact checks: muscle structure, not crosstalk or session artefacts?**

**Read first:** Tresch, Cheung & d'Avella 2006, *Matrix factorization algorithms for the identification of muscle synergies* (J Neurophysiol); Kutch & Valero-Cuevas 2012 (why surface EMG alone can't prove synergies are neural). **Owner:** Person C

**We expect:** the synergies from notebook 02 are the same across random starts, beat channel-shuffled data, explain held-out stages almost as well as their own, and spread wider than one muscle seen through crosstalk. **Wrong if:** random starts give different synergies, shuffled data does as well, or every synergy is as compact as a single source blurred by crosstalk.

Low-dimensional EMG can come from things that aren't synergies: chance, or signal leaking between neighbouring
electrodes. Each check below gives **one number and one figure**. NMF and VAF come from notebook 02's reference
([`ref02.py`](../02_how_many_synergies/ref02.py)); if you finished notebook 02 you may paste your own versions.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref03.py`](ref03.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** **Check 0 (stability)** is ✅ **Aim 1**: the proposal picks k by held-out error *and* stability across fits. Checks 1 to 4 are ➕ beyond the proposal, but they are what stage 2 of the investigation needs.

**Goal:** show the k synergies from notebook 02 are stable, not chance co-activation, shared across movements, and not just crosstalk between neighbouring electrodes.

**Why:** NMF will *always* return k synergies, even from noise. Each check removes one alternative explanation, so later stages (sharing, decoding) are about real structure.

**Milestones**
1. ✅ Stability: 10 random starts give the same synergies
2. Shuffle test: real data beats channel-shuffled data
3. New-movement test: synergies fitted on other stages still explain held-out stages
4. Crosstalk test: neighbour share judged against a permutation null and a single-source crosstalk reference
5. Two-hands comparison, reported as *simultaneous bilateral* similarity, not independent replication

**Gate (decides the next stage):** stable (lowest matched similarity ≥ 0.9), above shuffle at every k, borrowed/own ≥ 0.9 on held-out stages, and at least half the synergies spread wider than the single-source crosstalk reference.

**Done when:** a pass/fail row for every check in the Result gate, each with its null.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "03_are_synergies_real"
assert (_d / "ref03.py").exists(), f"{_d}/ref03.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref03", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref03 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (builds it with prep.py the first time) ─────────────
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):   # missing, or built by an older version of prep.py
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording
filename_of_rec = d["filename"]                   # (30,) file name: pairs the two simultaneous recordings

train, test = split == 0, split == 1
SEED = 0   # pass seed=SEED to every random start so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

### The variables you start with
`N` = number of frames in all 30 recordings together (105,484: about 35 minutes at 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float64 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float64 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |
| `filename_of_rec` | (30,) | str | file name per recording. Left and right recordings with the same block number were made **at the same time**: 30 recordings = **15 independent blocks** |

Letters used in the docstrings below: **T** = number of frames in the array you pass in, **k** = number of
synergies. A good habit: try each function on a tiny made-up array first (e.g. `np.random.rand(3, 16)`).

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Tools from notebook 02, and keeping it fast
This notebook fits NMF **many** times (10 random starts, then dozens of shuffles). One fit on all training
frames takes about 5 s, so doing everything at full size would take many minutes. Two shortcuts keep the
whole notebook to a minute or two:
- **Every 5th frame** (`STEP = 5`). Neighbouring frames are 20 ms apart and the envelope is smoothed at 10 Hz,
  so they are nearly copies of each other: keeping 1 in 5 loses little information.
- **200 rounds of NMF updates** instead of 300. Notebook 02's error curve is almost flat by then.

The shuffle test (Check 1) needs about 50 fits, so it goes further: every 10th frame and 100 rounds.
These shortcuts make each number slightly noisier, not biased. If a result is borderline, rerun that one
check at full size before you trust it.

In [ ]:
# NMF and VAF from notebook 02. Finished notebook 02? You may paste your own versions here instead.
from ref02 import nmf_fit, nmf_transform, vaf

K = 4            # replace with the k chosen in notebook 02 (Task 7, leave-one-stage-out)
STEP = 5         # use every 5th frame for the repeated fits
N_ITER = 200     # NMF update rounds per fit

---
## Task 1 · Compare two sets of synergies
Every check below asks "are these two sets of synergies the same?" Two problems make that harder than it sounds.

**1. How similar are two synergies?** Each synergy is a list of 16 electrode weights: a direction in a
16-dimensional space. The **cosine similarity** is the cosine of the angle between two such directions:

$$\cos(w_1, w_2) = \frac{w_1 \cdot w_2}{\lVert w_1\rVert\,\lVert w_2\rVert}$$

It is 1 when the two have the same *pattern* (whatever their size), and 0 when they share no electrodes.
NMF weights are never negative, so it can't go below 0 here. For two whole sets, `W1` (k × 16) and `W2`
(k × 16), make every row length 1; then **one matrix product** `W1 @ W2.T` gives all k × k similarities at once.

**2. Which synergy goes with which?** NMF returns synergies in a random order: synergy 1 in one run may be
synergy 3 in another. So pair them up first, so that the **total** similarity is as large as possible.
Trying every pairing is `k!` options (3.6 million for k = 10). The **Hungarian algorithm** solves it in about
`k³` steps. It is a long combinatorial algorithm with nothing to teach about synergies, so here we use SciPy's
`linear_sum_assignment` for this one step. It **minimises** a cost, so give it **minus** the similarity.

In [ ]:
from scipy.optimize import linear_sum_assignment


def cosine_similarity_matrix(W1: np.ndarray, W2: np.ndarray) -> np.ndarray:
    """All pairwise cosine similarities between two sets of synergies.

    Parameters
    ----------
    W1 : (k1, 16) float, one set of synergies (rows need not have length 1)
    W2 : (k2, 16) float, another set

    Returns
    -------
    S : (k1, k2) float, S[i, j] = cosine similarity of W1[i] and W2[j]
    Check: cosine_similarity_matrix(W, W) has 1.0 on the diagonal.
    """
    # TODO
    raise NotImplementedError


def match_synergies(W1: np.ndarray, W2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pair up two sets of synergies (they come out in random order) and compare each pair.

    Parameters
    ----------
    W1 : (k, 16) float, one set of synergies
    W2 : (k, 16) float, another set

    Returns
    -------
    sims  : (k,) float in [0, 1], cosine similarity of each matched pair (1 = identical)
    order : (k,) int, W2[order[i]] is the partner of W1[i]

    Steps: the (k, k) similarity matrix, then linear_sum_assignment (it minimises, so pass the negative).
    Check: match_synergies(W, W[::-1]) → sims all 1.0, order = [k-1, …, 1, 0]
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `cosine_similarity_matrix`: divide every row of `W1` by its length, do the same for `W2`, then one matrix
   product gives every pair's dot product, which for length-1 rows *is* the cosine similarity.
2. `match_synergies`: build the matrix `S`, run `linear_sum_assignment(-S)`. It returns two arrays,
   `rows` and `cols`: row `rows[i]` of `W1` is paired with column `cols[i]` of `W2`.
3. Read off each pair's similarity from `S`, and return the pairing as `order`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.norm(W, axis=1, keepdims=True)` → the length of each row, shape (k, 1), ready to divide by.
- `A @ B.T` → (k1, k2): entry [i, j] is the dot product of row i of A with row j of B.
- `linear_sum_assignment` returns `rows` already sorted as 0, 1, …, k−1 for a square matrix, so `order = cols`.
- `S[np.arange(k), order]` picks one entry per row: `S[0, order[0]]`, `S[1, order[1]]`, …

```python
S = np.array([[0.1, 0.9], [0.8, 0.2]])
rows, cols = linear_sum_assignment(-S)   # rows = [0, 1], cols = [1, 0]
S[np.arange(2), cols]                    # array([0.9, 0.8])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def cosine_similarity_matrix(W1, W2):
    A = W1 / np.linalg.norm(W1, axis=____, keepdims=True)
    B = W2 / np.linalg.norm(W2, axis=____, keepdims=True)
    return ____ @ ____.T


def match_synergies(W1, W2):
    S = cosine_similarity_matrix(W1, W2)
    rows, cols = linear_sum_assignment(____)
    order = cols
    return S[np.arange(len(order)), ____], order
```

</details>

In [ ]:
# ✅ Check Task 1
rng_toy = np.random.default_rng(0)
W_a, W_b = rng_toy.random((4, 16)), rng_toy.random((5, 16))
S = cosine_similarity_matrix(W_a, W_b)
assert S.shape == (4, 5), f"shape {S.shape}, expected (4, 5)"
assert np.allclose(S, ref.cosine_similarity_matrix(W_a, W_b)), "similarities differ: did you make each row length 1?"
assert np.allclose(np.diag(cosine_similarity_matrix(W_a, W_a)), 1), "a synergy should have similarity 1 with itself"
sims, order = match_synergies(W_a, W_a[::-1])
assert np.allclose(sims, 1) and list(order) == [3, 2, 1, 0], f"reversed copy: sims {sims}, order {order}"
W_c = rng_toy.random((4, 16))
r_sims, r_order = ref.match_synergies(W_a, W_c)
sims, order = match_synergies(W_a, W_c)
assert np.array_equal(order, r_order) and np.allclose(sims, r_sims), "pairing differs: pass MINUS the similarity"
print(f"✅ Matching works. Two random sets of synergies still score {sims.mean():.2f} on average: keep that in mind!")

**Notice that last number.** Random non-negative weights already have a cosine similarity of about 0.7–0.8,
because all-positive vectors always point roughly the same way. So "similarity 0.8" means little on its own:
every check below compares against a **null** (what you'd get by chance), not against 0.

✍️ **Q1.** Two sets of *random* non-negative synergies already score a high cosine similarity (the check above printed it).
Why do all-positive vectors always point roughly the same way? What does that mean for a statement like
"the synergies were 0.85 similar"?

> *Your answer:*

---
## Task 2 · A one-line wrapper: fit and return the synergies
Every check fits NMF and only needs `W`. Write a small wrapper so the rest of the notebook stays short.
`nmf_fit` already scales the rows of `W` to length 1; normalise again anyway, so the function also works with
an NMF that doesn't (e.g. scikit-learn's).

In [ ]:
def fit_synergies(X: np.ndarray, k: int, seed: int, n_iter: int = N_ITER) -> np.ndarray:
    """Fit NMF and return its synergies, each scaled to length 1.

    Parameters
    ----------
    X      : (T, 16) float, non-negative EMG (any subset of frames)
    k      : int, number of synergies
    seed   : int, the random start for nmf_fit
    n_iter : int, NMF update rounds

    Returns
    -------
    W : (k, 16) float, row i = synergy i, with np.linalg.norm(W[i]) == 1
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Call `nmf_fit` with the same `k`, `n_iter` and `seed`. It returns three things; keep only `W`.
2. Divide each row of `W` by its length and return it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `_, W, _ = nmf_fit(X, k, n_iter=n_iter, seed=seed)` keeps the middle value.
- `W / np.linalg.norm(W, axis=1, keepdims=True)` scales every row to length 1.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_synergies(X, k, seed, n_iter=N_ITER):
    _, W, _ = nmf_fit(____, ____, n_iter=n_iter, seed=____)
    return W / np.linalg.norm(W, axis=____, keepdims=True)
```

</details>

In [ ]:
# ✅ Check Task 2
X_small = emg[train][:3000]
W_try = fit_synergies(X_small, 3, seed=SEED, n_iter=50)
assert W_try.shape == (3, 16), f"shape {W_try.shape}, expected (3, 16)"
assert np.allclose(np.linalg.norm(W_try, axis=1), 1), "rows should have length 1"
W_ref = ref.fit_synergies(X_small, 3, seed=SEED, n_iter=50)
print(f"✅ fit_synergies works (matched similarity with the reference: {match_synergies(W_try, W_ref)[0].mean():.3f})")

✍️ **Q2.** Why scale every synergy to length 1 before comparing? What would go wrong in `match_synergies` if one fit returned
a synergy 10× longer than the other fit's version of it?

> *Your answer:*

---
## Check 0 · Stable across random starts (✅ in the proposal)
NMF starts from random numbers and climbs downhill to the nearest good answer. If the data has real structure,
every start should land on the **same** synergies; if it is noise, each start finds its own.

Fit NMF 10 times with seeds 0 … 9 (on every `STEP`-th training frame), then match each run against run 0.
**Figure:** one dot per run and synergy, matched similarity to run 0. **Pass if** all are close to 1
(a common bar in the literature is above about 0.9; agree yours as a team).

In [ ]:
N_RUNS = 10
X_fit = emg[train][::STEP]                    # (T_train / STEP, 16)

W_runs: list[np.ndarray] = []                 # length N_RUNS, each (K, 16)
# TODO: fit_synergies(X_fit, K, seed) for seed = 0 … N_RUNS − 1, append each W

stability = None                              # (N_RUNS − 1, K): match_synergies(W_runs[0], W_runs[r])[0] for r = 1 … 9
# TODO: fill stability; print its smallest value
# TODO: figure: x = run number 1 … 9, y = the K similarities of that run (one dot each), a dashed line at 0.9

#### 💡 Stuck on Check 0? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. A loop over seeds: fit, append.
2. A second loop over runs 1 … 9: match against run 0 and keep only the similarities. Stack them into one
   (9, K) array.
3. Plot: for each run, K dots at the same x position.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.array(list_of_arrays)` stacks a list of (K,) arrays into a (9, K) array.
- `ax.scatter(x, y)` with `x = np.repeat(np.arange(1, 10), K)` and `y = stability.ravel()` draws every dot at once.

```python
np.repeat(np.arange(1, 3), 2)        # array([1, 1, 2, 2])
np.array([[0.9, 1.0], [0.8, 0.95]]).ravel()   # array([0.9 , 1.  , 0.8 , 0.95])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for seed in range(N_RUNS):
    W_runs.append(fit_synergies(____, K, seed=____))

stability = np.array([match_synergies(W_runs[0], W_runs[r])[____] for r in range(1, N_RUNS)])
print(f"smallest matched similarity: {stability.____():.3f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(np.repeat(np.arange(1, N_RUNS), K), stability.ravel())
ax.axhline(0.9, ls="--", color="gray"); ax.set_ylim(0, 1.05)
ax.set_xlabel("run (compared with run 0)"); ax.set_ylabel("matched cosine similarity"); plt.show()
```

</details>

In [ ]:
# ✅ Check 0
assert len(W_runs) == N_RUNS and all(W.shape == (K, 16) for W in W_runs), "10 runs, each (K, 16)"
stability = np.asarray(stability)
assert stability.shape == (N_RUNS - 1, K), f"stability shape {stability.shape}, expected {(N_RUNS - 1, K)}"
assert ((stability >= 0) & (stability <= 1 + 1e-9)).all(), "similarities should be between 0 and 1"
assert np.allclose(W_runs[3], ref.fit_synergies(X_fit, K, seed=3)), "run 3 differs from fit_synergies(X_fit, K, seed=3)"
print(f"✅ Stability: smallest matched similarity {stability.min():.3f}, mean {stability.mean():.3f}")

✍️ **Q3.** Ten random starts gave nearly identical synergies. Does stability prove the synergies are *real*? Name one thing that
would be perfectly stable but not a muscle synergy.

> *Your answer:*

---
## ➕ Beyond the proposal · Check 1 · Shuffle test
**The idea.** A synergy means "these electrodes switch on **together**". To test whether that togetherness is
real, destroy it and see what is lost. Shift each channel's data in time by its **own** random amount
(`np.roll`, which wraps the end round to the start). Each channel keeps exactly its own values, so its
average level and how much it varies are unchanged, but which channels are active *at the same moment*
is scrambled.

Then do the notebook 02 experiment on both versions: fit on train, rebuild test, VAF for each k.
If the real curve is clearly above the shuffled one, co-activation carries real structure.

**What to expect.** Even shuffled data reaches a high VAF at k = 1: VAF is uncentered, so one synergy that
rebuilds each channel's *average level* already explains a lot. The **gap** between the curves is what
measures co-activation, not the height of either one.

**Settings:** every 10th frame, 100 NMF rounds, k = 1 … 8, 5 shuffles (about 50 fits, under a minute).

In [ ]:
def shuffle_channels(X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Shift each channel in time by its own random amount, breaking co-activation.

    Parameters
    ----------
    X   : (T, D) float, EMG
    rng : np.random.default_rng(seed), the source of the random shifts

    Returns
    -------
    X_shuf : (T, D) float, column j = np.roll(X[:, j], shifts[j])
    Draw all shifts at once, in this order, so your answer matches the reference:
        shifts = rng.integers(0, T, size=D)
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Check 1 · shuffle_channels? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Read `T` and `D` from `X.shape`.
2. Draw the `D` shifts in one call (the recipe in the docstring).
3. Roll each column by its own shift, and put the columns back side by side.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.roll(a, 2)` moves every value 2 places later and wraps the last 2 round to the start.
- `np.column_stack([col0, col1, …])` puts 1-D arrays side by side as columns.

```python
np.roll(np.array([1, 2, 3, 4]), 1)                 # array([4, 1, 2, 3])
np.column_stack([np.array([1, 2]), np.array([3, 4])])   # [[1, 3], [2, 4]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def shuffle_channels(X, rng):
    T, D = X.shape
    shifts = rng.integers(0, ____, size=____)
    return np.column_stack([np.roll(X[:, j], ____) for j in range(D)])
```

</details>

In [ ]:
# ✅ Check shuffle_channels
X_toy = np.arange(40, dtype=float).reshape(10, 4)
X_s = shuffle_channels(X_toy, np.random.default_rng(1))
assert X_s.shape == X_toy.shape, f"shape {X_s.shape}, expected {X_toy.shape}"
assert all(np.array_equal(np.sort(X_s[:, j]), X_toy[:, j]) for j in range(4)), "each column must keep its own values"
assert np.array_equal(X_s, ref.shuffle_channels(X_toy, np.random.default_rng(1))), \
    "the shifts differ from the reference: draw them with rng.integers(0, T, size=D) in one call"
print("✅ shuffle_channels works")

Now the experiment. For each k: fit on the (real or shuffled) training frames, rebuild the matching test
frames with `nmf_transform` (synergies fixed), and score with `vaf`. Shuffle train and test **separately**
(two calls), so the shuffled test data has no co-activation for the synergies to find either.

In [ ]:
ks_shuf = list(range(1, 9))
N_SHUFFLES = 5
STEP_SHUF, N_ITER_SHUF = 10, 100
X_tr = emg[train][::STEP_SHUF]
X_te = emg[test][::STEP_SHUF]

vaf_real: list[float] = []                              # length 8: test VAF per k on the real data
vaf_shuf = np.full((N_SHUFFLES, len(ks_shuf)), np.nan)  # (5 shuffles, 8 k values)
rng = np.random.default_rng(SEED)
# TODO: fill vaf_real: for each k, nmf_fit on X_tr (n_iter=N_ITER_SHUF, seed=SEED), nmf_transform X_te, vaf
# TODO: for each shuffle s: shuffle X_tr and X_te (two calls with the same rng), then fill row s of vaf_shuf the same way
# TODO: figure: vaf_real as a line, vaf_shuf as a shaded band (min to max over shuffles); print the gap at k = K

#### 💡 Stuck on Check 1 · the experiment? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Write a small helper `test_vaf(X_train, X_test, k)` that fits, transforms and returns the VAF. Both loops then
   become one line each.
2. Real: loop over `ks_shuf`, append to `vaf_real`.
3. Shuffled: for `s` in range(N_SHUFFLES), shuffle both arrays, then loop over `ks_shuf` and fill `vaf_shuf[s, i]`.
4. Plot with `ax.fill_between(ks_shuf, vaf_shuf.min(axis=0), vaf_shuf.max(axis=0), alpha=0.3)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `enumerate(ks_shuf)` gives `(i, k)` pairs, so you know which column to fill.
- `ax.fill_between(x, low, high)` shades the area between two curves.

```python
for i, k in enumerate([1, 2, 3]):
    print(i, k)       # 0 1 · 1 2 · 2 3
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def test_vaf(X_train, X_test, k):
    _, W, _ = nmf_fit(____, k, n_iter=N_ITER_SHUF, seed=SEED)
    C = nmf_transform(____, W, n_iter=N_ITER_SHUF, seed=SEED)
    return vaf(X_test, ____ @ ____)

vaf_real = [test_vaf(X_tr, X_te, k) for k in ks_shuf]
for s in range(N_SHUFFLES):
    tr_s, te_s = shuffle_channels(X_tr, rng), shuffle_channels(____, rng)
    for i, k in enumerate(ks_shuf):
        vaf_shuf[s, i] = test_vaf(____, ____, k)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ks_shuf, vaf_real, marker="o", label="real")
ax.fill_between(ks_shuf, vaf_shuf.min(axis=0), vaf_shuf.max(axis=0), alpha=0.3, label="shuffled")
ax.set_xlabel("k"); ax.set_ylabel("test VAF (%)"); ax.legend(); plt.show()
print(f"gap at k = {K}: {vaf_real[K - 1] - vaf_shuf[:, K - 1].mean():.1f} points")
```

</details>

In [ ]:
# ✅ Check 1
assert len(vaf_real) == len(ks_shuf), f"{len(vaf_real)} real VAFs, expected {len(ks_shuf)}"
assert not np.isnan(vaf_shuf).any(), "some entries of vaf_shuf were never filled"
_, r_W, _ = ref.ref02.nmf_fit(X_tr, K, n_iter=N_ITER_SHUF, seed=SEED)
r_real = ref.ref02.vaf(X_te, ref.ref02.nmf_transform(X_te, r_W, n_iter=N_ITER_SHUF, seed=SEED) @ r_W)
assert abs(vaf_real[K - 1] - r_real) < 0.5, f"real VAF at k = {K}: {vaf_real[K - 1]:.2f}%, reference {r_real:.2f}%"
gap = vaf_real[K - 1] - vaf_shuf[:, K - 1].mean()
print(f"✅ At k = {K}: real {vaf_real[K - 1]:.1f}% vs shuffled {vaf_shuf[:, K - 1].mean():.1f}% (gap {gap:.1f} points)")

✍️ **Q4.** The shuffled VAF is still high. Why (think about what shifting each channel keeps and what it destroys, and about
uncentred VAF)? Why is the **gap** between the curves the number to report, not the height of either curve?

> *Your answer:*

---
## ➕ Beyond the proposal · Check 2 · New-movement test
**The idea.** If synergies are building blocks the nervous system reuses, synergies learned from some tasks
should also explain **tasks they have never seen**. If instead NMF just memorises each task's patterns, they
won't.

Fit synergies on the training frames of 12 stages, then explain each of the other 3 stages with those
synergies **held fixed** (only the activations are found: `nmf_transform`). Compare with the best those stages
could do: synergies fitted on their **own** training frames. Score both on that stage's **test** frames.

**Figure:** a bar per held-out stage of `VAF borrowed / VAF own`. **Pass if** the ratio is close to 1
(e.g. above 0.95).

A ratio **above 1** is possible and not a bug: "own" synergies are fitted on one stage's training frames only
(two recordings), "borrowed" ones on twelve stages' worth, so the borrowed set can generalise better to the
stage's test frames.

**Which 3 stages?** The interesting choice is tasks that *differ* from the rest, e.g. a wrist task
(`WristFlexionAbduction`), a thumb task, a free-style task. Try a few sets and say which you report and why.

In [ ]:
stage_per_frame = stage_of_rec[rec]          # (N,) str
stages = np.unique(stage_of_rec)             # (15,) str
held_out = stages[:3]                        # TODO: choose 3 stages (and say why); try other choices too
is_held = np.isin(stage_per_frame, held_out) # (N,) bool

vaf_borrowed: dict[str, float] = {}          # stage → VAF (%)
vaf_own: dict[str, float] = {}               # stage → VAF (%)
# TODO: W_borrowed = fit_synergies(emg[train & ~is_held][::STEP], K, seed=SEED)
# TODO: for each held-out stage s:
#         X_s_tr = emg[train & (stage_per_frame == s)],  X_s_te = emg[test & (stage_per_frame == s)]
#         borrowed: C = nmf_transform(X_s_te, W_borrowed, seed=SEED); vaf of C @ W_borrowed
#         own:      W_own = fit_synergies(X_s_tr, K, seed=SEED); the same with W_own
# TODO: figure: a bar per held-out stage of vaf_borrowed[s] / vaf_own[s], with a dashed line at 1

#### 💡 Stuck on Check 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. One fit on the 12 training stages gives `W_borrowed`.
2. Loop over `held_out`. For each stage take its train and test frames with the two masks.
3. Score the test frames twice: with `W_borrowed`, and with `W_own` fitted on that stage's train frames.
   In both cases only the activations are found on the test frames.
4. Bars of the ratio, with stage names on the x axis.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Masks combine with `&`: `train & (stage_per_frame == s)` keeps frames that are both.
- `ax.bar(names, values)` labels each bar with a name; `ax.tick_params(axis="x", rotation=30)` tilts long names.

```python
a = np.array(["x", "y", "x"]); t = np.array([True, True, False])
t & (a == "x")        # array([ True, False, False])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
W_borrowed = fit_synergies(emg[train & ~is_held][::STEP], K, seed=SEED)

def vaf_with(X, W):
    C = nmf_transform(X, W, seed=SEED)
    return vaf(X, ____ @ ____)

for s in held_out:
    X_s_tr = emg[train & (stage_per_frame == s)]
    X_s_te = emg[____ & (stage_per_frame == s)]
    vaf_borrowed[s] = vaf_with(X_s_te, ____)
    vaf_own[s] = vaf_with(X_s_te, fit_synergies(____, K, seed=SEED))

ratios = [vaf_borrowed[s] / vaf_own[s] for s in held_out]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(held_out, ratios); ax.axhline(1, ls="--", color="gray"); ax.set_ylim(0.8, 1.05)
ax.set_ylabel("VAF borrowed / own"); ax.tick_params(axis="x", rotation=20); plt.show()
```

</details>

In [ ]:
# ✅ Check 2
assert set(vaf_borrowed) == set(vaf_own) == set(held_out), "one entry per held-out stage in both dicts"
s0 = held_out[0]
X0 = emg[test & (stage_per_frame == s0)]
r_Wb = ref.fit_synergies(emg[train & ~is_held][::STEP], K, seed=SEED)
r_b = ref.ref02.vaf(X0, ref.ref02.nmf_transform(X0, r_Wb, seed=SEED) @ r_Wb)
assert abs(vaf_borrowed[s0] - r_b) < 0.5, f"{s0}: borrowed VAF {vaf_borrowed[s0]:.2f}%, reference {r_b:.2f}% (fit on train & ~is_held, every STEP-th frame)"
print("✅ Borrowed / own VAF ratio: " + ", ".join(f"{s}: {vaf_borrowed[s] / vaf_own[s]:.3f}" for s in held_out))

✍️ **Q5.** A borrowed/own ratio above 1 is possible. How? Which three stages did you hold out, and would the answer change for
free-style stages, which mix many movements?

> *Your answer:*

---
## ➕ Beyond the proposal · Check 3 · Neighbour (crosstalk) test
**The worry.** Neighbouring electrodes partly record the *same* muscle, because electrical signal spreads
through the skin and fat (**crosstalk**). NMF would happily report "electrodes 4, 5 and 6 switch on together"
as a synergy, when it is really one muscle seen by three electrodes. A real synergy, several muscles working
together, should usually reach **beyond** one small patch of the band.

**The measure.** The 16 electrodes sit in a ring around the wrist (assumed from the channel order; check this
against the emg2pose paper). For each synergy: what share of its total weight sits on its strongest electrode
and the two next to it? Evenly spread weight gives 3/16 ≈ 19%; everything on one patch gives 100%.

**Figure:** a polar plot per synergy (16 weights around a circle, like the band). **Pass if** synergies span
several regions (shares well below 100%).

In [ ]:
def neighbour_share(w: np.ndarray) -> float:
    """Share of a synergy's weight on its strongest electrode and its two ring neighbours.

    Parameters
    ----------
    w : (16,) float, one synergy (non-negative)

    Returns
    -------
    float in [3/16, 1]. ≈ 0.19 = spread evenly; 1.0 = all on three adjacent electrodes.
    The band is a ring: the neighbours of electrode 0 are 15 and 1, those of 15 are 14 and 0.
    """
    # TODO
    raise NotImplementedError


W = None                    # (K, 16): TODO fit_synergies on emg[train][::STEP]
shares = None               # (K,) float: neighbour_share of each row
# TODO: polar plot per synergy: 16 angles around the circle, radius = weight

#### 💡 Stuck on Check 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Find the strongest electrode `i`.
2. Its neighbours are `i − 1` and `i + 1`, **wrapping round** the ring: use `% 16` so that −1 becomes 15 and
   16 becomes 0.
3. Add the three weights and divide by the sum of all 16.
4. Polar plot: one subplot per synergy with `subplot_kw={"projection": "polar"}`; angles `2π · j / 16`.
   Repeat the first point at the end so the shape closes.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.argmax(w)` → the index of the largest entry.
- `-1 % 16` is 15 and `16 % 16` is 0 in Python: the remainder always wraps into 0 … 15.
- `np.append(a, a[0])` adds the first value again at the end.

```python
w = np.array([5.0, 1.0, 0.0, 4.0])
i = int(np.argmax(w))                       # 0
(w[(i - 1) % 4] + w[i] + w[(i + 1) % 4]) / w.sum()   # (4 + 5 + 1) / 10 = 1.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def neighbour_share(w):
    i = int(np.argmax(w))
    n = len(w)
    return float((w[(i - 1) % ____] + w[i] + w[(i + 1) % ____]) / w.____())

W = fit_synergies(emg[train][::STEP], K, seed=SEED)
shares = np.array([neighbour_share(w) for w in ____])

angles = 2 * np.pi * np.arange(16) / 16
fig, ax = plt.subplots(1, K, figsize=(3.2 * K, 3.2), subplot_kw={"projection": "polar"})
for i in range(K):
    ax[i].plot(np.append(angles, angles[0]), np.append(W[i], W[i][0]))
    ax[i].set_title(f"synergy {i + 1}: {shares[i]:.0%}")
plt.tight_layout(); plt.show()
```

</details>

In [ ]:
# ✅ Check 3
assert abs(neighbour_share(np.ones(16)) - 3 / 16) < 1e-9, "even weights should give 3/16"
w_edge = np.zeros(16); w_edge[[15, 0, 1]] = 1
assert abs(neighbour_share(w_edge) - 1) < 1e-9, "weights on 15, 0, 1 are neighbours on the ring: share should be 1"
w_toy = np.random.default_rng(2).random(16)
assert abs(neighbour_share(w_toy) - ref.neighbour_share(w_toy)) < 1e-9
assert W.shape == (K, 16) and np.allclose(W, ref.fit_synergies(emg[train][::STEP], K, seed=SEED))
assert np.allclose(shares, [ref.neighbour_share(w) for w in W])
print("✅ Neighbour shares: " + ", ".join(f"{s:.0%}" for s in shares) + f"  (evenly spread = {3 / 16:.0%})")

### Check 3b · Judge the neighbour share against two references
A neighbour share means little on its own. Two references bracket it:
1. **Permutation null (is there any spatial layout?).** Keep a synergy's 16 weights but put them on random
   electrodes. That keeps how *concentrated* the synergy is, but destroys *where* the weights sit. If the real share
   is above the permuted shares, the weight really is spatially clustered.
2. **Single-source crosstalk reference (is it more than one muscle?).** What would **one** muscle look like across
   the ring? Its signal spreads to neighbouring electrodes, so the correlation between electrodes falls with ring
   distance and then flattens at a plateau (activity that is shared for other reasons). Subtract the plateau,
   rescale so distance 0 is 1, and lay the falloff around the ring: that is a single-source pattern. Its neighbour
   share is the share a synergy would have if it were **just one muscle plus crosstalk**.

The correlations also contain real co-activation of neighbouring muscles, so this single-source pattern is **wider**
than true crosstalk: a synergy with a share **below** its share spreads wider than even a generous single source.

In [ ]:
def permuted_shares(w: np.ndarray, rng: np.random.Generator, n_perm: int = 1000) -> np.ndarray:
    """Neighbour shares of n_perm random re-orderings of w's 16 weights.

    Parameters
    ----------
    w   : (16,) float, one synergy
    rng : np.random.default_rng(seed); call rng.permutation(w) once per re-ordering, in a loop

    Returns
    -------
    (n_perm,) float
    """
    # TODO
    raise NotImplementedError


def single_source_pattern(C: np.ndarray, far_from: int = 5) -> np.ndarray:
    """One source's spread around the ring, estimated from the channel correlation matrix.

    Parameters
    ----------
    C        : (16, 16) correlation matrix of the EMG channels
    far_from : ring distances >= far_from count as the far-away plateau

    Returns
    -------
    (16,) float ≥ 0, centred on electrode 0: entry j = bump at ring distance min(j, 16 − j), where
    bump[d] = max(0, (profile[d] − plateau) / (1 − plateau)), profile[d] = mean of C[i, (i + d) % 16] over i,
    plateau = mean of profile[far_from:]
    """
    # TODO
    raise NotImplementedError


C_ch = np.corrcoef(emg[train][::STEP].T)      # (16, 16) channel correlations
rng_perm = np.random.default_rng(SEED)
perm_p = None              # (K,) float: share of permutations scoring ≥ the real share, per synergy
single_share = None        # float: neighbour_share(single_source_pattern(C_ch))
wider_than_one = None      # (K,) bool: real share < single_share

#### 💡 Stuck on Check 3b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `permuted_shares`: in a loop, permute `w` with `rng.permutation(w)` and score it with `neighbour_share`.
2. `single_source_pattern`: (a) the mean correlation at each ring distance d = 0 … 8; (b) the plateau = mean of the
   far distances; (c) subtract, divide, clip at 0; (d) place it around the ring with `min(j, 16 − j)`.
3. Per synergy: `perm_p[i] = np.mean(permuted_shares(W[i], rng_perm) >= shares[i])`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `C[i, (i + d) % 16]` is the correlation between electrode i and the one d places round the ring.
- `np.clip(a, 0, None)` sets negatives to 0. `np.minimum(np.arange(16), 16 - np.arange(16))` → ring distance to 0.

```python
dist = np.minimum(np.arange(6), 6 - np.arange(6))    # [0, 1, 2, 3, 2, 1]
bump = np.array([1.0, 0.5, 0.1, 0.0])
bump[dist]                                          # [1. , 0.5, 0.1, 0. , 0.1, 0.5]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def permuted_shares(w, rng, n_perm=1000):
    return np.array([neighbour_share(rng.permutation(____)) for _ in range(n_perm)])

def single_source_pattern(C, far_from=5):
    n = C.shape[0]
    prof = np.array([np.mean([C[i, (i + d) % n] for i in range(n)]) for d in range(n // 2 + 1)])
    plateau = prof[____:].mean()
    bump = np.clip((prof - plateau) / (1 - ____), 0, None)
    return bump[np.minimum(np.arange(n), n - np.arange(n))]

perm_p = np.array([np.mean(permuted_shares(W[i], rng_perm) >= shares[i]) for i in range(K)])
single_share = neighbour_share(single_source_pattern(____))
wider_than_one = shares < ____
```

</details>

In [ ]:
# ✅ Check 3b
w_toy = np.random.default_rng(5).random(16)
ps = permuted_shares(w_toy, np.random.default_rng(0), 200)
assert ps.shape == (200,) and np.allclose(ps, ref.permuted_shares(w_toy, np.random.default_rng(0), 200)), \
    "call rng.permutation(w) once per re-ordering, in order"
assert np.allclose(single_source_pattern(C_ch), ref.single_source_pattern(C_ch)), "single-source pattern differs"
assert abs(single_share - ref.neighbour_share(ref.single_source_pattern(C_ch))) < 1e-9
assert perm_p.shape == (K,) and wider_than_one.dtype == bool
print(f"✅ single-source share {single_share:.0%}; real shares " + ", ".join(f"{s:.0%}" for s in shares)
      + "; permutation p " + ", ".join(f"{p:.3f}" for p in perm_p)
      + f"; {wider_than_one.sum()} of {K} synergies spread wider than one source")

### 📊 Real neighbour share vs the two references
Each bar is one synergy's share. Grey band: 5th–95th percentile of its permutation null. Red line: the single-source
crosstalk reference. **What to look for:** bars above the grey band (spatially clustered) and **below** the red line
(wider than one muscle plus crosstalk). Bars above the red line can't be told apart from a single source.

In [ ]:
null_lo, null_hi = zip(*[np.percentile(ref.permuted_shares(W[i], np.random.default_rng(SEED), 1000), [5, 95])
                         for i in range(K)])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
x = np.arange(1, K + 1)
ax[0].bar(x, null_hi, bottom=0, color="0.85", label="permutation null (95th pct)")
ax[0].bar(x, shares, width=0.4, color="C0", label="real share")
ax[0].axhline(single_share, color="C3", lw=2, label="single source + crosstalk")
ax[0].set_xlabel("synergy"); ax[0].set_ylabel("neighbour share"); ax[0].set_ylim(0, 1); ax[0].legend(fontsize=8)
ax[1].plot(range(16), single_source_pattern(C_ch), marker="o", color="C3")
ax[1].set_xlabel("electrode (ring, source at 0)"); ax[1].set_title("single-source pattern from channel correlations")
plt.tight_layout(); plt.show()

✍️ **Q6.** The synergies are more spatially compact than the permutation null, which is what crosstalk would produce, but also
what muscles that sit next to each other would produce. Which synergies spread wider than the single-source
reference, and which can't be told apart from one muscle seen through crosstalk? What measurement would separate
the two explanations?

> *Your answer:*

---
## ➕ Beyond the proposal · Check 4 · Two-hands test
**The idea.** If synergies reflect how the hand is built and controlled, the left and right hands should
have the same ones. Fit synergies on left-hand recordings and on right-hand recordings separately, then compare.

**The catch: the band doesn't sit the same way on both wrists.** On the other wrist the electrode ring runs
the **other way round** (a mirror image), and it may be put on **rotated** by a few electrodes. So before
comparing, try every way the right-hand band could line up with the left: 16 rotations, each normal or
reversed = 32 versions. Keep the best.

**Why we need a null.** Trying 32 versions and keeping the best always flatters the score, even for unrelated
synergies (and random non-negative synergies already score about 0.75, as Task 1 showed). So run the **same
search** for 100 sets of random non-negative synergies.

**⚠️ What this check can and can't show: the two hands were recorded at the same time.** The left and right
recordings of each block share a block number in `filename_of_rec` and overlap 100% in time. In most blocks both
hands do the same task together; in one block (`OneHandedFreeStyle`) one hand moves while the other hand's recording
(`unconstrained`) is mostly idle. So the hands are **not independent replications**: a high score can come from the
two hands doing the same movement at the same moment. Report the score as **similarity during simultaneous
bilateral movement**. A real replication needs different sessions or other people (stage 7).
**Pass if** the real score is clearly above the null, *read with that caveat*.

In [ ]:
def channel_variants(W: np.ndarray) -> list[np.ndarray]:
    """Every way the other wrist's band could line up: 16 rotations × (normal, reversed).

    Parameters
    ----------
    W : (k, 16) float, synergies from one hand

    Returns
    -------
    list of 32 arrays, each (k, 16): the same synergies with the 16 columns rotated and/or reversed.
    Order (to match the reference): the 16 rotations of W, then the 16 rotations of W reversed.
    """
    # TODO
    raise NotImplementedError


def best_match(W_left: np.ndarray, W_right: np.ndarray) -> float:
    """Mean matched cosine similarity, using the best of the 32 channel_variants of W_right.

    W_left, W_right : (k, 16). Returns a float in [0, 1].
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Check 4 · channel_variants and best_match? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `channel_variants`: two versions of the band, `W` as it is and `W` with its columns in reverse order.
   For each, rotate the columns by 0, 1, …, 15 places. That is 2 × 16 = 32 arrays.
2. `best_match`: for every variant, match it against `W_left` and take the mean similarity. Return the largest.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `W[:, ::-1]` reverses the order of the columns (electrode 15 first).
- `np.roll(W, s, axis=1)` rotates the columns by `s` places (rows untouched).
- `max(...)` of a generator: `max(f(v) for v in items)`.

```python
W = np.array([[1, 2, 3]])
W[:, ::-1]                  # [[3, 2, 1]]
np.roll(W, 1, axis=1)       # [[3, 1, 2]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def channel_variants(W):
    out = []
    for Wv in (W, W[:, ::-1]):
        for s in range(W.shape[1]):
            out.append(np.roll(____, ____, axis=1))
    return out


def best_match(W_left, W_right):
    return max(float(match_synergies(W_left, V)[0].____()) for V in channel_variants(____))
```

</details>

In [ ]:
# ✅ Check channel_variants and best_match
W_toy = np.random.default_rng(3).random((3, 16))
variants = channel_variants(W_toy)
assert len(variants) == 32 and all(v.shape == (3, 16) for v in variants), "32 variants, each (k, 16)"
for i, v in enumerate(ref.channel_variants(W_toy)):
    assert any(np.array_equal(v, u) for u in variants), f"reference variant {i} is missing from your list"
W_rot = np.roll(W_toy[:, ::-1], 5, axis=1)[::-1]    # reversed, rotated by 5 and listed in another order
assert abs(best_match(W_toy, W_rot) - 1) < 1e-9, "a reversed, rotated, reordered copy should score 1.0"
W_other = np.random.default_rng(4).random((3, 16))
assert abs(best_match(W_toy, W_other) - ref.best_match(W_toy, W_other)) < 1e-9
print("✅ channel_variants and best_match work")

Now the real comparison. Left-hand and right-hand recordings come in pairs from the same tasks, so both sides
saw the same movements.

In [ ]:
side_per_frame = side_of_rec[rec]              # (N,) str
N_NULL = 100
rng_null = np.random.default_rng(SEED)

W_left  = None   # (K, 16): TODO fit_synergies on emg[train & (side_per_frame == "left")][::STEP]
W_right = None   # (K, 16): the same for "right"
score_real = None                              # float: best_match(W_left, W_right)
score_null: list[float] = []                   # N_NULL floats: best_match(W_left, random synergies)
# TODO: fill all four. A random set of synergies: rng_null.random((K, 16)), then make each row length 1
# TODO: print score_real and the 95th percentile of score_null

#### 💡 Stuck on Check 4 · the comparison? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Two fits, one per hand, with the side mask combined with `train`.
2. `score_real` is one call of `best_match`.
3. The null: 100 times, draw random non-negative weights, normalise the rows, and score them against `W_left`
   with the **same** `best_match` search.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.percentile(score_null, 95)` → the value 95% of the null scores fall below. If the real score beats it,
  chance alone rarely does that well.

```python
R = np.random.default_rng(0).random((2, 3))
R / np.linalg.norm(R, axis=1, keepdims=True)    # each row now has length 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
W_left = fit_synergies(emg[train & (side_per_frame == "left")][::STEP], K, seed=SEED)
W_right = fit_synergies(emg[train & (side_per_frame == ____)][::STEP], K, seed=SEED)
score_real = best_match(____, ____)
for _ in range(N_NULL):
    R = rng_null.random((K, 16))
    score_null.append(best_match(W_left, R / np.linalg.norm(R, axis=____, keepdims=True)))
print(f"real {score_real:.3f} | null 95th percentile {np.percentile(score_null, 95):.3f}")
```

</details>

In [ ]:
# ✅ Check 4
assert W_left.shape == W_right.shape == (K, 16)
assert np.allclose(W_left, ref.fit_synergies(emg[train & (side_per_frame == "left")][::STEP], K, seed=SEED))
assert abs(score_real - ref.best_match(W_left, W_right)) < 1e-9, "score_real should be best_match(W_left, W_right)"
assert len(score_null) == N_NULL and all(0 <= s <= 1 + 1e-9 for s in score_null)
beats = np.mean(np.array(score_null) >= score_real)
print(f"✅ Left vs right: {score_real:.3f}; null mean {np.mean(score_null):.3f}, "
      f"95th percentile {np.percentile(score_null, 95):.3f}; {beats:.0%} of null sets score at least as high")

### 📊 Left vs best-aligned right, and the null
Left: the K × K similarity matrix after re-aligning the right hand's band and pairing the synergies (a strong
diagonal = each left synergy has a right-hand twin). Right: where the real score sits among the null scores.

In [ ]:
V_best = max(channel_variants(W_right), key=lambda V: match_synergies(W_left, V)[0].mean())
_, order = match_synergies(W_left, V_best)
S_lr = cosine_similarity_matrix(W_left, V_best[order])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(S_lr, vmin=0, vmax=1, cmap="viridis")
ax[0].set_xlabel("right-hand synergy (aligned, paired)"); ax[0].set_ylabel("left-hand synergy")
plt.colorbar(im, ax=ax[0], label="cosine similarity")
ax[1].hist(score_null, bins=20, color="0.7", label="random synergies (null)")
ax[1].axvline(score_real, color="C3", lw=2, label="left vs right")
ax[1].set_xlabel("best matched similarity"); ax[1].legend()
plt.tight_layout(); plt.show()

✍️ **Q7.** The best of the 32 band alignments in Check 4 was "no change at all". Give two explanations (think about how the
dataset may store the left-hand channels, and about the hands being recorded at the same time). What would you need
to check before claiming the two hands share synergies?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **15 independent blocks, not 30 recordings** | Left and right recordings of a block were made at the same time (blocks 14–15 pair OneHandedFreeStyle with the idle hand's 'unconstrained') | Count blocks, not recordings or frames, when judging how sure a result is |
| **Two hands are simultaneous** | Check 4 can't separate 'same synergies' from 'same movement at the same moment' | Compare hands across different sessions, or across people (stage 7) |
| **Speed shortcuts** | Every 5th/10th frame and fewer NMF rounds add noise to every number | Rerun any borderline check at full size |
| **Uncentred VAF in Check 1–2** | Shuffled and borrowed VAFs look high partly because of the mean | Judge gaps and ratios, or recompute with centred VAF (notebook 02) |
| **Crosstalk reference is approximate** | The single-source pattern comes from channel correlations, which include real co-activation, so it is wider than true crosstalk | Differential electrode pairs, or comparing with anatomy of the forearm under the band |
| **Ring order assumed** | Neighbour share and band rotations assume channels 0–15 run in order around the wrist | Check the emg2pose paper's electrode layout |
| **Surface EMG can't prove a neural origin** | Stable, shared, non-crosstalk components can still come from biomechanics (Kutch & Valero-Cuevas 2012) | Only perturbation or neural recordings can; word conclusions as necessary conditions |

---
## Result · gate for stage 2
**Question:** Are the synergies stable, beyond chance, shared across movements, and wider than crosstalk?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| 0 · Stability: lowest matched similarity over 10 starts (✅ Aim 1) | | | |
| 1 · Shuffle: real − shuffled VAF gap at k | | | |
| 2 · New movements: borrowed / own VAF per held-out stage | | | |
| 3 · Crosstalk: shares vs permutation p and vs single-source share | | | |
| 4 · Two hands (simultaneous): best match vs null 95th percentile | | | |

**Gate:** checks 0–2 pass and at least half the synergies spread wider than the single-source reference (check 4 is reported, not gated, because the hands are simultaneous).

- **Pass →** stage 3 (notebook 08): do stages reuse these synergies in different combinations?
- **Fail →** say which alternative explanation survives (chance, crosstalk) and stop calling the components synergies; stage 4 can still ask what the EMG encodes.
- **Was your prediction right? Why or why not?**